# FAQ Chatbot Analysis and Implementation (V5.0)
This notebook analyzes the `customer_support_faq.csv` dataset and implements the optimized **SentenceTransformers (all-MiniLM-L6-v2) RAG** system with half-precision floating-point embeddings for resource optimization.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load the Data
df = pd.read_csv('customer_support_faq.csv')
print(f"Dataset contains {df.shape[0]} questions across {df['category'].nunique()} categories.")
df.head()

## 1. Exploratory Data Analysis
Let's see the distribution of questions per category.

In [ ]:
df['category'].value_counts().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Number of Questions per Category')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Chatbot V5.0 Semantic Search Implementation
To handle semantic variations and make the chatbot more robust against different phrasings (which TF-IDF struggles with), we use a dense vector embedding approach via **SentenceTransformers (all-MiniLM-L6-v2)**.

Furthermore, we optimize this system for V5.0:
1. **Semantic Embeddings**: Convert questions into 384-dimensional dense vectors.
2. **Float16 Quantization**: Store the vectors as half-precision (`float16`) to halve the model storage size.
3. **NumPy Cosine Similarity**: Compute cosine similarities directly using NumPy, avoiding PyTorch dependency during runtime.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the SentenceTransformer model
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings and convert them to numpy float16
print("Creating embeddings for FAQs...")
faq_embeddings = embedder.encode(df['question'].tolist(), convert_to_numpy=True).astype(np.float16)
print(f"Embeddings shape: {faq_embeddings.shape}, type: {faq_embeddings.dtype}")

def get_chatbot_response(user_query, threshold=0.4):
    """
    Takes a user query, encodes it, computes cosine similarity using NumPy,
    and returns the most relevant FAQ answer if similarity exceeds threshold.
    """
    # Encode the user query (convert to float32 for calculations)
    query_vec = embedder.encode(user_query, convert_to_numpy=True).astype(np.float32)
    corpus_vecs = faq_embeddings.astype(np.float32)
    
    # Normalize vectors for cosine similarity calculation
    query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-8)
    corpus_norms = corpus_vecs / (np.linalg.norm(corpus_vecs, axis=1, keepdims=True) + 1e-8)
    
    # Compute similarities
    similarities = corpus_norms @ query_norm.T
    
    # Get best match
    best_match_idx = similarities.argmax()
    
    if similarities[best_match_idx] >= threshold:
        best_question = df.iloc[best_match_idx]['question']
        answer = df.iloc[best_match_idx]['answer']
        return f"**Matched FAQ:** {best_question} (Score: {similarities[best_match_idx]:.3f})\n\n**Answer:** {answer}"
    else:
        return "I'm sorry, I couldn't find a relevant answer to your question. Please try rephrasing or contact support."

## 3. Testing the Chatbot
Let's test the chatbot with a few sample queries.

In [ ]:
test_queries = [
    "How can I track where my order is?",
    "I want to return an item I got as a gift without a receipt",
    "What happens if I forgot my password?",
    "Can you tell me a joke?"  # Expected to fail the threshold
]

for q in test_queries:
    print(f"\033[94mUser Query:\033[0m {q}")
    print(f"\033[92mChatbot:\033[0m {get_chatbot_response(q)}")
    print("-"*60)

## 4. Model Export (V5.0)
To use this optimized model in our FastAPI engine, we export the FAQ DataFrame and the float16 embeddings as a `.joblib` file.

In [ ]:
import joblib
import os

# Export the optimized RAG data
model_data = {
    'df': df,
    'embeddings': faq_embeddings,
    'format_version': 2
}

joblib.dump(model_data, 'chatbot_rag_data.joblib')
print('Model successfully optimized and exported to chatbot_rag_data.joblib')
print(f"File size: {os.path.getsize('chatbot_rag_data.joblib') / 1024:.1f} KB")